In [3]:
import os

from datasets import load_dataset

# base_dir = Path("/home/bsprenge/.cache/huggingface/lerobot/bensprenger/pusht")

# dataset = load_dataset("parquet", data_files=os.path.join(base_dir, "*", "*.parquet"))
# # Print the dataset structure.
# print(dataset)
# # Show a few examples, where you'll see the 'episode' column derived from the folder names.
# print("First row from the dataset:", dataset['train'][0])
# print("Second row from the dataset:", dataset['train'][1])


# Define the base directory for our partitioned dataset.
base_dir = "/home/bsprenge/.cache/huggingface/lerobot/bensprenger/my_partitioned_dataset"
# Create folders for each episode.
for episode in [0, 1]:
    os.makedirs(os.path.join(base_dir, f"episode={episode}"), exist_ok=True)
# Create sample data for each episode.

import pandas as pd

df_episode0 = pd.DataFrame({"value": [10, 20, 30]})
df_episode1 = pd.DataFrame({"value": [40, 50, 60]})
# Convert the dataframes to PyArrow Tables.
import pyarrow as pa
import pyarrow.parquet as pq

table0 = pa.Table.from_pandas(df_episode0)
table1 = pa.Table.from_pandas(df_episode1)
# # Write the tables to Parquet files inside the partition folders.
pq.write_table(table0, os.path.join(base_dir, "episode=0", "data.parquet"))
pq.write_table(table1, os.path.join(base_dir, "episode=1", "data.parquet"))
print("Partitioned dataset created in", base_dir)

Partitioned dataset created in /home/bsprenge/.cache/huggingface/lerobot/bensprenger/my_partitioned_dataset


In [4]:
base_dir

'/home/bsprenge/.cache/huggingface/lerobot/bensprenger/my_partitioned_dataset'

In [ ]:
dataset = load_dataset("parquet", data_files=os.path.join(base_dir, "*", "*.parquet"), partitioning="hive")

TypeError: ParquetConfig.__init__() got an unexpected keyword argument 'partitioning'

In [12]:
import numpy as np

table = pa.table({"a": range(10), "b": np.random.randn(10), "c": [1, 2] * 5, "part": ["a"] * 5 + ["b"] * 5})

pq.write_to_dataset(
    table, "/home/bsprenge/.cache/tmp_partition_test/parquet_dataset_partitioned", partition_cols=["part"]
)

In [14]:
import pyarrow.dataset as ds

dataset = ds.dataset(
    "/home/bsprenge/.cache/tmp_partition_test/parquet_dataset_partitioned",
    format="parquet",
    partitioning="hive",
)

In [17]:
print(dataset.files)
dataset.to_table().to_pandas().head(3)

['/home/bsprenge/.cache/tmp_partition_test/parquet_dataset_partitioned/part=a/da63326845aa4ecf91203e4ba07c1642-0.parquet', '/home/bsprenge/.cache/tmp_partition_test/parquet_dataset_partitioned/part=b/da63326845aa4ecf91203e4ba07c1642-0.parquet']


,a,b,c,part
0,0,0.680264,1,a
1,1,0.966352,2,a
2,2,0.358236,1,a


In [37]:
new_ds = load_dataset(
    "parquet",
    data_files=os.path.join(
        "/home/bsprenge/.cache/tmp_partition_test/parquet_dataset_partitioned", "*", "*.parquet"
    ),
    storage_options={
        "partitioning": "hive",
        "partition_base_dir": "/home/bsprenge/.cache/tmp_partition_test/parquet_dataset_partitioned",
    },
)

In [38]:
new_ds["train"].to_pandas().head(3)

,a,b,c
0,0,0.680264,1
1,1,0.966352,2
2,2,0.358236,1


In [20]:
import datasets.Dataset

ModuleNotFoundError: No module named 'datasets.Dataset'

In [22]:
ds.Dataset.load_from_disk

AttributeError: type object 'pyarrow._dataset.Dataset' has no attribute 'load_from_disk'

In [23]:
import datasets

In [26]:
datasets.Dataset.load_from_disk("/home/bsprenge/.cache/huggingface/lerobot/bensprenger/pusht")

FileNotFoundError: No such files: '/home/bsprenge/.cache/huggingface/lerobot/bensprenger/pusht/dataset_info.json', nor '/home/bsprenge/.cache/huggingface/lerobot/bensprenger/pusht/state.json' found. Expected to load a `Dataset` object but provided path is not a `Dataset`.

In [36]:
os.path.join("/home/bsprenge/.cache/tmp_partition_test/parquet_dataset_partitioned", "*", "*.parquet")

'/home/bsprenge/.cache/tmp_partition_test/parquet_dataset_partitioned/*/*.parquet'

In [ ]:
ds = load_dataset(
    "parquet",
    data_files="path/to/dataset/**/*.parquet",
    parquet_read_options={"use_pandas_metadata": True, "partitioning": "hive"},
)

TypeError: ParquetConfig.__init__() got an unexpected keyword argument 'hf_load_kwargs'

In [ ]:
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

Help on function load_hf_dataset in module lerobot.common.datasets.lerobot_dataset:

load_hf_dataset(self) -> datasets.arrow_dataset.Dataset
    TODO profile



In [17]:
dataset = LeRobotDataset("bensprenger/aloha_mobile_cabinet")

In [19]:
dataset.hf_dataset[10000]

{'observation.state': tensor([-0.4264, -0.3528,  0.7624, -0.4111, -0.3068,  0.0230,  0.0319,  0.3022,
         -0.3820,  0.7701, -0.4264, -0.0890,  0.6673,  0.0473]),
 'observation.effort': tensor([  45.7300, -360.4600, -529.9300,   24.2100, -118.3600,   -2.6900,
          -99.5300,  147.9500, -443.8500, -470.7500,    0.0000,  -10.7600,
          -80.7000,  -86.0800]),
 'action': tensor([-0.3988, -0.3774,  0.7501, -0.4203, -0.2424,  0.0414,  0.0481,  0.3267,
         -0.4264,  0.7609, -0.4280, -0.0506,  0.6121,  0.0715]),
 'episode_index': tensor(6),
 'frame_index': tensor(1000),
 'timestamp': tensor(20.),
 'next.done': tensor(False),
 'index': tensor(10000),
 'task_index': tensor(0)}

In [21]:
from datasets import Dataset

In [22]:
ds = Dataset.from_parquet(
    str(dataset.root / "data" / "chunk-000" / "episode_index=0" / "episode_000000.parquet")
)

Generating train split: 0 examples [00:00, ? examples/s]